# BA and Pooled-Region Stress Event Catalog

This notebook demonstrates the five-step historical (2007–2023) stress-event workflow for the two BAs used in Figures 10–11 (`SWPP` and `MISO_8910`), the pooled `MISO_SUBREGION_SUM` region used by the weather-context example, and `WECC` as a fourth example. The same functions apply to every region in the packaged manifests.

Load and net load use pooled 90th, 95th, and 99th percentiles. Renewable-equivalent capacity factor uses common absolute 10%, 5%, and 1% thresholds. Risk hours separated by one non-risk hour are bridged into one event; the skipped hour is recorded in `gap_hours`.

## Shared Five-Step Workflow

`scenario metrics → event table → quantile ranks → thresholds → risk hours → stress events`

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path("outputs/ba_stress_event_catalog")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_COLUMNS = [
    "ba_code",
    "metric",
    "threshold_quantile",
    "event_start_utc",
    "event_end_utc",
    "event_length_hours",
    "gap_hours",
    "event_peak_value",
    "event_peak_quantile",
    "scenario_specific_event_peak_quantile",
    "event_peak_utc",
]



In [2]:
# Step 1: Convert scenario metrics to the event-pipeline table.

def scenario_metrics_to_event_table(scenario_metrics, scenario_metric_metadata):
    """Convert scenario-metrics rows into the event-pipeline table for isolated BAs and BA pools."""
    scenario_metrics = scenario_metrics.assign(time_utc=pd.to_datetime(scenario_metrics["time_utc"], utc=True))
    event_table_frames = []

    for _, scenario_metric_metadata_row in scenario_metric_metadata.iterrows():
        catalog_metric = scenario_metric_metadata_row["metric"]
        scenario = scenario_metric_metadata_row["scenario"]
        scenario_metrics_value_column = scenario_metric_metadata_row["risk_metric_type"]

        scenario_rows = scenario_metrics.loc[scenario_metrics["scenario"].eq(scenario), ["time_utc", scenario_metrics_value_column]]
        if scenario_rows.empty:
            raise AssertionError(f"Missing scenario rows for {catalog_metric}: {scenario}")

        event_rows = scenario_rows.rename(columns={scenario_metrics_value_column: "value"})
        event_rows["metric"] = catalog_metric
        event_table_frames.append(event_rows[["metric", "time_utc", "value"]])

    if not event_table_frames:
        raise AssertionError("No event-table rows were created.")

    return pd.concat(event_table_frames, ignore_index=True)


In [3]:
# Step 2: Calculate quantile ranks.

def calculate_quantile_ranks(hourly_load_netload_eqcf_values, risk_metric_settings):
    """Add risk-type settings and quantile-rank context for one region."""

    # Attach the visible settings table to each hourly row using the exact metric name.
    hourly_load_netload_eqcf_values = hourly_load_netload_eqcf_values.merge(risk_metric_settings, on="metric", how="left")

    # Require explicit settings for every metric included in the event table.
    unmatched_metrics = hourly_load_netload_eqcf_values.loc[hourly_load_netload_eqcf_values["risk_metric_type"].isna(), "metric"].unique()
    if len(unmatched_metrics) > 0:
        raise ValueError(f"Some metrics did not match risk_metric_settings: {sorted(unmatched_metrics)}")

    # Below for loops rank each metric series. The first loop ranks by each risk_metric_type, the second loop ranks by each metric.  
    for risk_metric_type, risk_type_values in hourly_load_netload_eqcf_values.groupby("risk_metric_type"):
        tie_rank_method = risk_type_values["tie_rank_method"].iloc[0]
        hourly_load_netload_eqcf_values.loc[risk_type_values.index, "event_peak_quantile"] = risk_type_values["value"].rank(method=tie_rank_method, pct=True)  

    for metric, metric_values in hourly_load_netload_eqcf_values.groupby("metric"):
        tie_rank_method = metric_values["tie_rank_method"].iloc[0]
        hourly_load_netload_eqcf_values.loc[metric_values.index, "scenario_specific_event_peak_quantile"] = metric_values["value"].rank(method=tie_rank_method, pct=True)  # scenario_specific_event_peak_quantile uses only the event's original metric series.

    return hourly_load_netload_eqcf_values


In [4]:
# Step 3: Calculate thresholds.

def calculate_thresholds(hourly_load_netload_eqcf_values):
    """Calculate quantile load thresholds and absolute renewable-CF thresholds."""
    threshold_table_rows = []

    # Calculate one shared threshold table for each risk metric type.
    for risk_metric_type, risk_type_values in hourly_load_netload_eqcf_values.groupby("risk_metric_type"):
        threshold_quantiles = risk_type_values["threshold_quantiles"].iloc[0]
        for threshold_quantile in threshold_quantiles:
            threshold_table_rows.append({
                "risk_metric_type": risk_metric_type,
                "threshold_quantile": threshold_quantile,
                "threshold_value": threshold_quantile if risk_metric_type == "renew_cf_equiv" else risk_type_values["value"].quantile(threshold_quantile),
            })

    return pd.DataFrame(threshold_table_rows)


In [5]:
# Step 4: Identify risk hours.

def identify_risk_hours(hourly_load_netload_eqcf_values, load_netload_eqcf_thresholds):
    """Return hourly rows that cross each threshold for one region."""
    risk_hour_columns = ["metric", "risk_metric_type", "threshold_quantile", "threshold_value", "time_utc", "value", "event_peak_quantile", "scenario_specific_event_peak_quantile"]

    # Attach each risk_metric_type threshold to every hourly row in that risk group.
    candidate_risk_hours = hourly_load_netload_eqcf_values.merge(load_netload_eqcf_thresholds, on="risk_metric_type")

    # Apply Eq. 5-style threshold-relative stress.
    # Load/net-load are upper-tail risks; renewable-CF-equivalent is a lower-tail risk.
    is_load_netload = candidate_risk_hours["risk_metric_type"].isin(["load_mw", "net_load_mw"])
    is_renewable_cf = candidate_risk_hours["risk_metric_type"].eq("renew_cf_equiv")
    is_load_netload_risk_hour = is_load_netload & candidate_risk_hours["value"].ge(candidate_risk_hours["threshold_value"])
    is_renewable_cf_risk_hour = is_renewable_cf & candidate_risk_hours["value"].le(candidate_risk_hours["threshold_value"])
    is_risk_hour = is_load_netload_risk_hour | is_renewable_cf_risk_hour
    load_netload_eqcf_risk_hours = candidate_risk_hours.loc[is_risk_hour, risk_hour_columns].copy()
    return load_netload_eqcf_risk_hours.sort_values(["metric", "threshold_quantile", "time_utc"]).reset_index(drop=True)


In [6]:
# Step 5: Build the stress-event catalog.

def build_stress_event_catalog(risk_hours, region_code):
    """Build final stress-event catalog rows for one region."""
    if risk_hours.empty:
        return pd.DataFrame(columns=EVENT_COLUMNS)

    one_hour = pd.Timedelta(hours=1)
    risk_groups = ["metric", "threshold_quantile"]

    # 5a. Measure the time between consecutive risk hours.
    risk_hours = risk_hours.sort_values(risk_groups + ["time_utc"]).copy()
    risk_hours["time_gap"] = risk_hours.groupby(risk_groups)["time_utc"].diff()

    # 5b. Bridge one skipped hour and assign event IDs.
    risk_hours["new_event"] = (
        risk_hours["time_gap"].isna()
        | risk_hours["time_gap"].gt(2 * one_hour)
    )
    risk_hours["event_id"] = risk_hours.groupby(risk_groups)["new_event"].cumsum()
    event_groups = risk_groups + ["event_id"]

    # 5c. Calculate event spans and record bridged gap hours.
    risk_hours["gap_hour_text"] = (
        (risk_hours["time_utc"] - one_hour)
        .dt.strftime("%Y-%m-%dT%H:%M:%SZ")
        .where(risk_hours["time_gap"].eq(2 * one_hour))
    )

    stress_events = risk_hours.groupby(event_groups, as_index=False).agg(
        event_start_utc=("time_utc", "first"),
        event_end_utc=("time_utc", "last"),
        gap_hours=("gap_hour_text", lambda hours: ";".join(hours.dropna())),
    )

    # Add 1 because an event beginning and ending at 05:00 lasts one hour.
    stress_events["event_length_hours"] = (
        (stress_events["event_end_utc"] - stress_events["event_start_utc"])
        / one_hour
    ).astype(int) + 1

    # 5d. Select the most stressful hour in each event.
    # Higher load is stressful; lower renewable CF is stressful.
    peak_direction = risk_hours["risk_metric_type"].map(
        {"load_mw": 1, "net_load_mw": 1, "renew_cf_equiv": -1}
    )
    risk_hours["peak_score"] = risk_hours["value"] * peak_direction

    # Because risk_hours is time-sorted, idxmax keeps the earliest tied peak.
    peak_indexes = risk_hours.groupby(event_groups)["peak_score"].idxmax()
    event_peaks = risk_hours.loc[
        peak_indexes,
        event_groups
        + [
            "value",
            "event_peak_quantile",
            "scenario_specific_event_peak_quantile",
            "time_utc",
        ],
    ].rename(
        columns={"value": "event_peak_value", "time_utc": "event_peak_utc"}
    )

    stress_events = stress_events.merge(event_peaks, on=event_groups)
    stress_events["ba_code"] = region_code

    return (
        stress_events[EVENT_COLUMNS]
        .sort_values(["metric", "threshold_quantile", "event_start_utc"])
        .reset_index(drop=True)
    )


## Five-step toy example

In [7]:

# Step 1: Start from an event table constructed directly for the toy example.
# The historical workflow below creates the same table from scenario metrics.
# The wind/solar split labels are illustrative scenario names for the tiny example.
example_event_table = pd.DataFrame(
    [
        ("2023-01-01T00:00:00Z", "load_mw", 80),
        ("2023-01-01T00:00:00Z", "net_load_mw__wind25_solar75", 26),
        ("2023-01-01T00:00:00Z", "net_load_mw__wind50_solar50", 24),
        ("2023-01-01T00:00:00Z", "renew_cf_equiv__wind25_solar75", 0.54),
        ("2023-01-01T00:00:00Z", "renew_cf_equiv__wind50_solar50", 0.56),
        ("2023-01-01T01:00:00Z", "load_mw", 95),
        ("2023-01-01T01:00:00Z", "net_load_mw__wind25_solar75", 65),
        ("2023-01-01T01:00:00Z", "net_load_mw__wind50_solar50", 63),
        ("2023-01-01T01:00:00Z", "renew_cf_equiv__wind25_solar75", 0.30),
        ("2023-01-01T01:00:00Z", "renew_cf_equiv__wind50_solar50", 0.32),
        ("2023-01-01T02:00:00Z", "load_mw", 96),
        ("2023-01-01T02:00:00Z", "net_load_mw__wind25_solar75", 95),
        ("2023-01-01T02:00:00Z", "net_load_mw__wind50_solar50", 94),
        ("2023-01-01T02:00:00Z", "renew_cf_equiv__wind25_solar75", 0.01),
        ("2023-01-01T02:00:00Z", "renew_cf_equiv__wind50_solar50", 0.02),
        ("2023-01-01T03:00:00Z", "load_mw", 84),
        ("2023-01-01T03:00:00Z", "net_load_mw__wind25_solar75", 75),
        ("2023-01-01T03:00:00Z", "net_load_mw__wind50_solar50", 70),
        ("2023-01-01T03:00:00Z", "renew_cf_equiv__wind25_solar75", 0.09),
        ("2023-01-01T03:00:00Z", "renew_cf_equiv__wind50_solar50", 0.14),
        ("2023-01-01T04:00:00Z", "load_mw", 97),
        ("2023-01-01T04:00:00Z", "net_load_mw__wind25_solar75", 95),
        ("2023-01-01T04:00:00Z", "net_load_mw__wind50_solar50", 93),
        ("2023-01-01T04:00:00Z", "renew_cf_equiv__wind25_solar75", 0.02),
        ("2023-01-01T04:00:00Z", "renew_cf_equiv__wind50_solar50", 0.04),
        ("2023-01-01T05:00:00Z", "load_mw", 98),
        ("2023-01-01T05:00:00Z", "net_load_mw__wind25_solar75", 97),
        ("2023-01-01T05:00:00Z", "net_load_mw__wind50_solar50", 96),
        ("2023-01-01T05:00:00Z", "renew_cf_equiv__wind25_solar75", 0.01),
        ("2023-01-01T05:00:00Z", "renew_cf_equiv__wind50_solar50", 0.02),
    ],
    columns=["time_utc", "metric", "value"],
)
example_event_table["time_utc"] = pd.to_datetime(example_event_table["time_utc"], utc=True)


example_risk_metric_settings = pd.DataFrame(
    [
        ("load_mw", "load_mw", [0.75, 0.90], "max"),
        ("net_load_mw__wind25_solar75", "net_load_mw", [0.75, 0.90], "max"),
        ("net_load_mw__wind50_solar50", "net_load_mw", [0.75, 0.90], "max"),
        ("renew_cf_equiv__wind25_solar75", "renew_cf_equiv", [0.25, 0.10], "min"),
        ("renew_cf_equiv__wind50_solar50", "renew_cf_equiv", [0.25, 0.10], "min"),
    ],
    columns=["metric", "risk_metric_type", "threshold_quantiles", "tie_rank_method"],
)

# Step 2: Calculate quantile ranks.
example_ranked = calculate_quantile_ranks(example_event_table, example_risk_metric_settings)

# Step 3: Calculate thresholds.
example_thresholds = calculate_thresholds(example_ranked)

# Step 4: Identify risk hours.
example_risk_hours = identify_risk_hours(example_ranked, example_thresholds)

# Step 5: Build the stress-event catalog.
example_catalog = build_stress_event_catalog(example_risk_hours, region_code="A")

print("Example Thresholds")
display(example_thresholds)
print("Example Risk Hours")
display(example_risk_hours)
print("Example Event Catalog")
display(example_catalog.head(8))

Example Thresholds


,risk_metric_type,threshold_quantile,threshold_value
0,load_mw,0.75,96.75
1,load_mw,0.90,97.50
2,net_load_mw,0.75,95.00
3,net_load_mw,0.90,95.90
4,renew_cf_equiv,0.25,0.25
5,renew_cf_equiv,0.10,0.10


Example Risk Hours


,metric,risk_metric_type,threshold_quantile,threshold_value,time_utc,value,event_peak_quantile,scenario_specific_event_peak_quantile
0,load_mw,load_mw,0.75,96.75,2023-01-01 04:00:00+00:00,97.00,0.833333,0.833333
1,load_mw,load_mw,0.75,96.75,2023-01-01 05:00:00+00:00,98.00,1.000000,1.000000
2,load_mw,load_mw,0.90,97.50,2023-01-01 05:00:00+00:00,98.00,1.000000,1.000000
3,net_load_mw__wind25_solar75,net_load_mw,0.75,95.00,2023-01-01 02:00:00+00:00,95.00,0.833333,0.833333
4,net_load_mw__wind25_solar75,net_load_mw,0.75,95.00,2023-01-01 04:00:00+00:00,95.00,0.833333,0.833333
5,net_load_mw__wind25_solar75,net_load_mw,0.75,95.00,2023-01-01 05:00:00+00:00,97.00,1.000000,1.000000
6,net_load_mw__wind25_solar75,net_load_mw,0.90,95.90,2023-01-01 05:00:00+00:00,97.00,1.000000,1.000000
7,net_load_mw__wind50_solar50,net_load_mw,0.75,95.00,2023-01-01 05:00:00+00:00,96.00,0.916667,1.000000
8,net_load_mw__wind50_solar50,net_load_mw,0.90,95.90,2023-01-01 05:00:00+00:00,96.00,0.916667,1.000000
9,renew_cf_equiv__wind25_solar75,renew_cf_equiv,0.10,0.10,2023-01-01 02:00:00+00:00,0.01,0.083333,0.166667


Example Event Catalog


,ba_code,metric,threshold_quantile,event_start_utc,event_end_utc,event_length_hours,gap_hours,event_peak_value,event_peak_quantile,scenario_specific_event_peak_quantile,event_peak_utc
0,A,load_mw,0.75,2023-01-01 04:00:00+00:00,2023-01-01 05:00:00+00:00,2,,98.00,1.000000,1.000000,2023-01-01 05:00:00+00:00
1,A,load_mw,0.90,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,98.00,1.000000,1.000000,2023-01-01 05:00:00+00:00
2,A,net_load_mw__wind25_solar75,0.75,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,2023-01-01T03:00:00Z,97.00,1.000000,1.000000,2023-01-01 05:00:00+00:00
3,A,net_load_mw__wind25_solar75,0.90,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,97.00,1.000000,1.000000,2023-01-01 05:00:00+00:00
4,A,net_load_mw__wind50_solar50,0.75,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,96.00,0.916667,1.000000,2023-01-01 05:00:00+00:00
5,A,net_load_mw__wind50_solar50,0.90,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,96.00,0.916667,1.000000,2023-01-01 05:00:00+00:00
6,A,renew_cf_equiv__wind25_solar75,0.10,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,,0.01,0.083333,0.166667,2023-01-01 02:00:00+00:00
7,A,renew_cf_equiv__wind25_solar75,0.25,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,,0.01,0.083333,0.166667,2023-01-01 02:00:00+00:00


## Historical inputs

The worked run uses `SWPP`, `MISO_8910`, `MISO_SUBREGION_SUM`, and `WECC` as examples. The historical loop calls the same five functions in the same order as the toy example; only its Step 1 must reshape the source scenario metrics into the event table. Change `REGION_CODES` to run the same procedure for another manifest-listed region.

In [8]:
SOURCE_BUNDLE = "wtk_bchrrr_nsrdb"
PERIOD_LABEL = "2007_2023"
REGION_CODES = ["SWPP", "MISO_8910", "MISO_SUBREGION_SUM", "WECC"]

ba_manifest = pd.read_csv(Path("../manifests/ba_scenario_metric_metadata_manifest_wtk_bchrrr_nsrdb_2007_2023.csv"))
pooled_manifest = pd.read_csv(Path("../manifests/pooled_region_scenario_metric_metadata_manifest_wtk_bchrrr_nsrdb_2007_2023.csv"))
metadata = pd.concat([ba_manifest, pooled_manifest], ignore_index=True)

RISK_METRIC_SETTINGS = pd.DataFrame(
    [
        ("load_mw", "load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__installed_2024", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w00_s100", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w25_s75", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w50_s50", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w75_s25", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w100_s00", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("renew_cf_equiv__installed_2024", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w00_s100", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w25_s75", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w50_s50", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w75_s25", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w100_s00", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
    ],
    columns=["metric", "risk_metric_type", "threshold_quantiles", "tie_rank_method"],
)

metadata = metadata.merge(RISK_METRIC_SETTINGS[["metric", "risk_metric_type"]], on="metric", how="left")
metadata = metadata[metadata["ba_code"].isin(REGION_CODES)].copy()
missing = sorted(set(REGION_CODES) - set(metadata["ba_code"]))
if missing:
    raise AssertionError(f"Missing manifest rows for {missing}.")
display(metadata.groupby("ba_code").agg(metrics=("metric", "size"), source=("source_path", "first")))

,metrics,source
ba_code,,
MISO_8910,13,data/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_me...
SWPP,13,data/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_me...
WECC,13,data/wtk_bchrrr_nsrdb_2007_2023/pooled_scenari...


## Run the Same Five Steps for the Historical Regions

In [9]:
def count_gap_hours(value):
    return 0 if value == "" else len(str(value).split(";"))


catalog_frames = []
validation_rows = []

for region_code in REGION_CODES:
    region_metadata = metadata[metadata["ba_code"].eq(region_code)].copy()
    source_path = Path("..") / region_metadata["source_path"].iloc[0]
    scenario_metrics = pd.read_csv(
        source_path,
        usecols=["time_utc", "scenario", "load_mw", "net_load_mw", "renew_cf_equiv"],
    )

    # Step 1: Convert scenario metrics to the event table.
    region_event_table = scenario_metrics_to_event_table(scenario_metrics, region_metadata)

    # Step 2: Calculate quantile ranks.
    region_ranked = calculate_quantile_ranks(region_event_table, RISK_METRIC_SETTINGS)

    # Step 3: Calculate thresholds.
    region_thresholds = calculate_thresholds(region_ranked)

    # Step 4: Identify risk hours.
    region_risk_hours = identify_risk_hours(region_ranked, region_thresholds)

    # Step 5: Build the stress-event catalog.
    region_catalog = build_stress_event_catalog(region_risk_hours, region_code=region_code)
    if region_catalog.empty:
        raise AssertionError(f"No events were created for {region_code}.")


    #Write results to CSV
    output_path = OUTPUT_DIR / f"{region_code}_{SOURCE_BUNDLE}_{PERIOD_LABEL}_events.csv"
    region_catalog.to_csv(output_path, index=False, date_format="%Y-%m-%dT%H:%M:%SZ")

    risk_hour_counts = region_risk_hours["risk_metric_type"].value_counts()
    validation_rows.append({
        "region": region_code,
        "hourly_metric_rows": len(region_event_table),
        "load_risk_hours": int(risk_hour_counts.get("load_mw", 0)),
        "net_load_risk_hours": int(risk_hour_counts.get("net_load_mw", 0)),
        "renewable_cf_risk_hours": int(risk_hour_counts.get("renew_cf_equiv", 0)),
        "events": len(region_catalog),
        "bridged_events": int(region_catalog["gap_hours"].ne("").sum()),
    })
    catalog_frames.append(region_catalog)
    del scenario_metrics, region_event_table, region_ranked, region_thresholds, region_risk_hours

catalog = pd.concat(catalog_frames, ignore_index=True)
validation = pd.DataFrame(validation_rows)
display(validation)
display(catalog[EVENT_COLUMNS].head(10))

,region,hourly_metric_rows,risk_hours,events,bridged_events
0,SWPP,1935960,455701,57160,811
1,MISO_8910,1935960,941366,122585,12746
2,WECC,1935960,430984,57731,505


,ba_code,metric,threshold_quantile,event_start_utc,event_end_utc,event_length_hours,gap_hours,event_peak_value,event_peak_quantile,scenario_specific_event_peak_quantile,event_peak_utc
0,SWPP,load_mw,0.9,2007-01-16 14:00:00+00:00,2007-01-16 16:00:00+00:00,3,,40914.39,0.920924,0.920924,2007-01-16 15:00:00+00:00
1,SWPP,load_mw,0.9,2007-01-17 15:00:00+00:00,2007-01-17 15:00:00+00:00,1,,39502.33,0.900047,0.900047,2007-01-17 15:00:00+00:00
2,SWPP,load_mw,0.9,2007-02-14 15:00:00+00:00,2007-02-14 16:00:00+00:00,2,,39763.16,0.904170,0.904170,2007-02-14 15:00:00+00:00
3,SWPP,load_mw,0.9,2007-02-15 14:00:00+00:00,2007-02-15 16:00:00+00:00,3,,40278.99,0.911946,0.911946,2007-02-15 15:00:00+00:00
4,SWPP,load_mw,0.9,2007-06-06 19:00:00+00:00,2007-06-07 01:00:00+00:00,7,,42762.85,0.943480,0.943480,2007-06-06 22:00:00+00:00
5,SWPP,load_mw,0.9,2007-06-07 19:00:00+00:00,2007-06-08 00:00:00+00:00,6,,42237.29,0.937503,0.937503,2007-06-07 22:00:00+00:00
6,SWPP,load_mw,0.9,2007-06-11 19:00:00+00:00,2007-06-12 01:00:00+00:00,7,,42668.98,0.942385,0.942385,2007-06-11 22:00:00+00:00
7,SWPP,load_mw,0.9,2007-06-12 18:00:00+00:00,2007-06-13 01:00:00+00:00,8,,42375.43,0.939108,0.939108,2007-06-12 22:00:00+00:00
8,SWPP,load_mw,0.9,2007-06-13 20:00:00+00:00,2007-06-13 23:00:00+00:00,4,,40250.10,0.911523,0.911523,2007-06-13 21:00:00+00:00
9,SWPP,load_mw,0.9,2007-06-14 20:00:00+00:00,2007-06-14 23:00:00+00:00,4,,40410.75,0.913914,0.913914,2007-06-14 22:00:00+00:00


## Appendix: State-level application

The deposited state catalogs use `load_mw__raw`, `net_load_mw__raw`, and renewable-equivalent capacity factor. These raw series retain TELL/TAIESM hourly weather variability without GCAM annual-demand scaling; the complete raw and eight GCAM-scaled trajectories remain in the packaged state scenario metrics. `state_load_generation.ipynb` and Appendix A of `site_cf_generation_ba_weighting_validation.ipynb` document the corresponding inputs.

If the GCAM trajectories are later used to select comparable stress periods, use one fixed raw-reference P95 across the cases rather than recalculating a percentile for each GCAM case. This preserves the effect of long-term load growth instead of normalizing it away. The fixed threshold is a diagnostic reference, not an adequacy standard.